In [19]:
import pandas as pd
from dotenv import load_dotenv
import os

import snowflake.connector

load_dotenv()

account= os.getenv("SNOWFLAKE_ACCOUNT")
user= os.getenv("SNOWFLAKE_USER")
password= os.getenv("SNOWFLAKE_PASSWORD")
warehouse= os.getenv("SNOWFLAKE_WAREHOUSE")
database= os.getenv("SNOWFLAKE_DATABASE")

connection= snowflake.connector.connect(account= account, user= user, password= password, warehouse= warehouse, database= database)

print(connection)
print("Connection successful:", connection.account)

Connection successful: ui48522


In [20]:
cursor= connection.cursor()

cursor.execute('SELECT * FROM HOUSING_AFFORDABILITY.MARTS.MART_AFFORDABILITY')
mart= cursor.fetchall()

In [21]:
columns= [col[0] for col in cursor.description]

In [22]:
df_mart= pd.DataFrame(data= mart, columns= columns)

df_mart

,MONTH_DATE,CITY,COMPOSITE_BENCHMARK,COMPOSITE_BENCHMARK_CHANGE,MORTGAGE_ARREARS,LOAN_TO_INCOME_RATIO,DEBT_SERVICE_RATIO,MORTGAGE_RATE_5YRS_FIXED,MORTGAGE_RATE_5YRS_VARIABLE,POLICY_RATE,POLICY_RATE_CHANGE,BENCHMARK_TO_RATE_RATIO
0,2018-01-01,CALGARY,420700,NaN,0.2,280.79,16.9,3.19,2.55,1.25,NaN,336560.000000
1,2018-01-01,EDMONTON,345000,NaN,0.2,280.79,16.9,3.19,2.55,1.25,NaN,276000.000000
2,2018-01-01,GREATER_TORONTO,749600,NaN,0.2,280.79,16.9,3.19,2.55,1.25,NaN,599680.000000
3,2018-01-01,GREATER_VANCOUVER,1029400,NaN,0.2,280.79,16.9,3.19,2.55,1.25,NaN,823520.000000
4,2018-01-01,HALIFAX_DARTMOUTH,273300,NaN,0.2,280.79,16.9,3.19,2.55,1.25,NaN,218640.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
779,2026-02-01,GREATER_VANCOUVER,1100300,-1600.0,NaN,NaN,NaN,3.99,3.70,2.25,0.0,489022.222222
780,2026-02-01,HALIFAX_DARTMOUTH,558600,13400.0,NaN,NaN,NaN,3.99,3.70,2.25,0.0,248266.666667
781,2026-02-01,MONTREAL_CMA,594200,14300.0,NaN,NaN,NaN,3.99,3.70,2.25,0.0,264088.888889
782,2026-02-01,OTTAWA,615400,8700.0,NaN,NaN,NaN,3.99,3.70,2.25,0.0,273511.111111


In [23]:
df_mart.describe()

,COMPOSITE_BENCHMARK,COMPOSITE_BENCHMARK_CHANGE,MORTGAGE_ARREARS,LOAN_TO_INCOME_RATIO,DEBT_SERVICE_RATIO,MORTGAGE_RATE_5YRS_FIXED,MORTGAGE_RATE_5YRS_VARIABLE,POLICY_RATE,POLICY_RATE_CHANGE,BENCHMARK_TO_RATE_RATIO
count,7.840000e+02,776.000000,256.000000,200.000000,256.000000,784.000000,784.000000,784.000000,776.000000,7.840000e+02
mean,5.830120e+05,1780.541237,0.173750,290.581200,17.822812,3.595102,3.554592,2.234694,0.010309,7.507080e+05
std,2.757632e+05,10456.020101,0.034957,16.871048,1.707854,1.110508,1.649793,1.653518,0.251236,1.034545e+06
min,2.664000e+05,-49700.000000,0.120000,267.870000,15.720000,1.690000,1.300000,0.250000,-1.500000,6.610000e+04
25%,3.674750e+05,-2500.000000,0.140000,279.010000,16.260000,2.690000,2.250000,0.500000,0.000000,1.602000e+05
50%,5.050000e+05,1100.000000,0.185000,284.110000,17.005000,3.565000,2.950000,1.750000,0.000000,2.425270e+05
75%,7.147250e+05,5700.000000,0.202500,306.740000,19.570000,4.590000,5.010000,3.750000,0.000000,1.150400e+06
max,1.279800e+06,75100.000000,0.220000,318.900000,20.600000,5.720000,6.300000,5.000000,1.000000,5.119200e+06


In [24]:
df_mart.shape

(784, 12)

In [25]:
df_mart.dtypes

MONTH_DATE                      object
CITY                            object
COMPOSITE_BENCHMARK              int64
COMPOSITE_BENCHMARK_CHANGE     float64
MORTGAGE_ARREARS               float64
LOAN_TO_INCOME_RATIO           float64
DEBT_SERVICE_RATIO             float64
MORTGAGE_RATE_5YRS_FIXED       float64
MORTGAGE_RATE_5YRS_VARIABLE    float64
POLICY_RATE                    float64
POLICY_RATE_CHANGE             float64
BENCHMARK_TO_RATE_RATIO        float64
dtype: object

In [26]:
connection.close()

In [27]:
df_mart= df_mart.sort_values(['CITY','MONTH_DATE'])
df_mart['BENCHMARK_YOY_PERCENTAGE_CHANGE']= df_mart.groupby('CITY')['COMPOSITE_BENCHMARK'].transform(lambda x: (x - x.shift(12))/x.shift(12) * 100)

df_mart['BENCHMARK_YOY_PERCENTAGE_CHANGE']= df_mart['BENCHMARK_YOY_PERCENTAGE_CHANGE'].round(2)
df_mart['BENCHMARK_TO_RATE_RATIO']= df_mart['BENCHMARK_TO_RATE_RATIO'].round(2)

In [28]:
df_mart.head(100)

,MONTH_DATE,CITY,COMPOSITE_BENCHMARK,COMPOSITE_BENCHMARK_CHANGE,MORTGAGE_ARREARS,LOAN_TO_INCOME_RATIO,DEBT_SERVICE_RATIO,MORTGAGE_RATE_5YRS_FIXED,MORTGAGE_RATE_5YRS_VARIABLE,POLICY_RATE,POLICY_RATE_CHANGE,BENCHMARK_TO_RATE_RATIO,BENCHMARK_YOY_PERCENTAGE_CHANGE
0,2018-01-01,CALGARY,420700,NaN,0.2,280.79,16.90,3.19,2.55,1.25,NaN,336560.00,NaN
8,2018-02-01,CALGARY,423100,2400.0,NaN,NaN,NaN,3.24,2.50,1.25,0.0,338480.00,NaN
16,2018-03-01,CALGARY,425500,2400.0,NaN,NaN,NaN,3.24,2.50,1.25,0.0,340400.00,NaN
24,2018-04-01,CALGARY,426800,1300.0,0.2,267.87,16.17,3.24,2.50,1.25,0.0,341440.00,NaN
32,2018-05-01,CALGARY,426900,100.0,NaN,NaN,NaN,3.34,2.45,1.25,0.0,341520.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
760,2025-12-01,CALGARY,554800,-2900.0,NaN,NaN,NaN,4.14,3.75,2.25,0.0,246577.78,-3.24
768,2026-01-01,CALGARY,555500,700.0,NaN,NaN,NaN,4.14,3.70,2.25,0.0,246888.89,-3.24
776,2026-02-01,CALGARY,562000,6500.0,NaN,NaN,NaN,3.99,3.70,2.25,0.0,249777.78,-2.72
1,2018-01-01,EDMONTON,345000,NaN,0.2,280.79,16.90,3.19,2.55,1.25,NaN,276000.00,NaN


In [31]:
df_mart.to_excel('..\data\processed\df_mart.xlsx')

In [32]:
df_mart['MONTH_DATE']= pd.to_datetime(df_mart['MONTH_DATE'])

start_date = '2019-01-01'
end_date = '2024-01-01'

df_model= df_mart.loc[(df_mart['MONTH_DATE'] >= start_date) & (df_mart['MONTH_DATE'] <= end_date)]

In [33]:
df_model.shape

(488, 13)

In [34]:
df_model['DEBT_SERVICE_RATIO'].isna().sum()

np.int64(320)

In [35]:
df_model= df_model.dropna(subset=['DEBT_SERVICE_RATIO'])

df_model.shape

(168, 13)

In [36]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error

results= []

for city in df_model['CITY'].unique():
    df_city= df_model.loc[df_model['CITY'] == city]
    
    X= df_city[['POLICY_RATE', 'BENCHMARK_YOY_PERCENTAGE_CHANGE']]
    #'LOAN_TO_INCOME_RATIO' omitted as the results were significant without it's addition
    Y= df_city['DEBT_SERVICE_RATIO']
    model= LinearRegression()
    model.fit(X, Y)
    policy_coef= model.coef_[0]
    y_pred= model.predict(X)
    r2= r2_score(Y, y_pred)
    rmse= root_mean_squared_error(Y, y_pred)
    results.append({
        'CITY': city,
        'POLICY_COEF': policy_coef,
        'R2': r2,
        'RMSE': rmse,
    })

df_results= pd.DataFrame(results)
df_results

,CITY,POLICY_COEF,R2,RMSE
0,CALGARY,0.918548,0.982665,0.229356
1,EDMONTON,1.050462,0.969072,0.306355
2,GREATER_TORONTO,1.109719,0.961404,0.342232
3,GREATER_VANCOUVER,1.026479,0.978936,0.252823
4,HALIFAX_DARTMOUTH,1.084019,0.964545,0.328011
5,MONTREAL_CMA,1.160616,0.961189,0.343185
6,OTTAWA,0.996853,0.939247,0.429371
7,WINNIPEG,1.092640,0.964555,0.327963
